# GraphKV revision v2: audit first
Read START_HERE.md. Use one T4. This notebook does not claim GPU validation. Run the cache gate before any pilot. Keep Kaggle Internet enabled and this notebook private if it contains sensitive paths. No credentials go in the output folder.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
PROJECT = Path("/kaggle/working/graphkv_revision_v2")
OUT = Path("/kaggle/working/graphkv_v2_hotpot")
assert (PROJECT / "run_revision.py").exists(), "Extract the package under /kaggle/working first"
os.chdir(PROJECT)
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True)


## Environment
Reuse the working installation if it exists. This installation cell is opt-in; do not mix nightly torch/audio wheels into the environment.

In [ ]:
INSTALL = False
if INSTALL:
    run("-m", "pip", "install", "-r", "requirements-revision.txt")
run("-m", "unittest", "discover", "-s", "tests", "-p", "test_revision.py", "-v")


## Optional verified Google Drive backup
First create a Google Drive rclone remote called gdrive on your laptop. Follow https://rclone.org/drive/ and use your own OAuth client. Put its full configuration in a private Kaggle Secret named RCLONE_CONFIG_TEXT. Install the official rclone binary in the runtime. Never print the config or save it inside OUT. Set ENABLE_DRIVE=True only after setup.

In [ ]:
ENABLE_DRIVE = False
if ENABLE_DRIVE:
    import shutil
    from kaggle_secrets import UserSecretsClient
    assert shutil.which("rclone"), "Install rclone first"
    conf = Path("/kaggle/working/private-rclone.conf")
    conf.write_text(UserSecretsClient().get_secret("RCLONE_CONFIG_TEXT"))
    conf.chmod(0o600)
    os.environ["RCLONE_CONFIG"] = str(conf)
    os.environ["GRAPHKV_BACKUP_REMOTE"] = "gdrive:GraphKV_Backups/v2_hotpot_run01"


## Prepare and measure selection drift (CPU)
This prepares 60 synthetic controlled test events. These are not production RAG traces. Use a separate OUT for each model/dataset. Preparation stores the corpus and similarity matrix for reproducibility. The quality evaluator below uses structured QA instead.

In [ ]:
run("run_revision.py", "prepare", "--output-dir", OUT, "--dataset", "hotpot",
    "--model", "Qwen/Qwen2.5-1.5B-Instruct", "--events", "60")
run("run_revision.py", "drift", "--output-dir", OUT, "--top-m", "5", "10", "20",
    "--k", "3", "6", "10")


## Mandatory GPU reuse gate
Six fresh process launches compare three cold/warm pairs. Inspect the report. Missing numerical or retrieval evidence is a stop, not a pass. Send reuse_gate.json and associated logs for review before expanding.

In [ ]:
run("run_revision.py", "gate", "--output-dir", OUT)
gate = json.loads((OUT / "gate/reuse_gate.json").read_text())
print("Reuse gate passed:", gate["passed"])
assert gate["passed"]


## Optional execution smoke after reviewing the gate
Disabled by default. This tests plumbing, not statistical significance. It runs six fresh blocks. Changing the matrix later requires a new experiment folder. We will design the larger pilot using these measured timings.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    run("run_revision.py", "run", "--output-dir", OUT, "--top-m", "20", "--k", "6",
        "--policies", "no_prefetch", "cosine", "two_hop", "--modes", "async",
        "--lead-ms", "0", "--block-events", "30", "--repetitions", "1")


## Optional full-prompt local answer quality
This starts a plain vLLM server: no LMCache connector or independent-chunk splicing. It measures EM/F1, not caching speedup. Stop the systems runner first. Choose hotpot, 2wiki, or musique. --include-legacy-fixed measures actual old/new fixed-policy answer-quality differences.

In [ ]:
RUN_QUALITY = False
if RUN_QUALITY:
    run("run_quality_managed.py", "--dataset", "hotpot", "--gpu", "0",
        "--model", "Qwen/Qwen2.5-1.5B-Instruct",
        "--output-dir", "/kaggle/working/quality_v2_hotpot",
        "--questions", "60", "--top-m", "5", "--k", "10", "--include-legacy-fixed")


## Recovery
Download the latest verified timestamped snapshot from Drive. Restore to an empty directory with restore_snapshot.py, then rerun the identical command. Completed systems blocks skip; the interrupted block reruns cold. Quality resumes at the first missing answer. A sudden shutdown can lose work since the last verified upload; no zero-loss guarantee is possible. Preserve the same source package and environment.

Example shell commands (replace the snapshot name):
```bash
rclone copyto gdrive:GraphKV_Backups/v2_hotpot_run01/SNAPSHOT.zip /kaggle/working/recovered.zip
python restore_snapshot.py /kaggle/working/recovered.zip /kaggle/working/graphkv_v2_hotpot
```